<a href="https://colab.research.google.com/github/harship123-source/Pfizer-Advanced-AI-Powered-Document-Insights-Data-Extraction-Externship/blob/main/Gradio_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import gradio as gr
import time, json, tempfile
import os

PURPLE = "#8b5cf6"
DARK   = "#0a0a0a"
DARKER = "#1a1a1a"

theme = gr.themes.Base(primary_hue="violet").set(
    # page + text
    body_background_fill="#000000",        body_background_fill_dark="#000000",
    body_text_color="#ffffff",             body_text_color_dark="#ffffff",
    body_text_color_subdued="#b3b3b3",     body_text_color_subdued_dark="#b3b3b3",
    background_fill_primary=DARK,          background_fill_primary_dark=DARK,
    background_fill_secondary=DARKER,      background_fill_secondary_dark=DARKER,
    # boxes
    block_background_fill=DARK,            block_background_fill_dark=DARK,
    block_border_color=PURPLE,             block_border_color_dark=PURPLE,
    block_border_width="2px",
    # label tabs ("Chat History", "Upload PDFs")
    block_label_background_fill=DARK,      block_label_background_fill_dark=DARK,
    block_label_text_color=PURPLE,         block_label_text_color_dark=PURPLE,
    block_label_border_color=PURPLE,       block_label_border_color_dark=PURPLE,
    # inputs
    input_background_fill="#111111",       input_background_fill_dark="#111111",
    input_border_color=PURPLE,             input_border_color_dark=PURPLE,
    input_placeholder_color="#888888",     input_placeholder_color_dark="#888888",
    # file list rows
    table_even_background_fill=DARK,       table_even_background_fill_dark=DARK,
    table_odd_background_fill=DARKER,      table_odd_background_fill_dark=DARKER,
    table_row_focus=DARKER,                table_row_focus_dark=DARKER,
    link_text_color="#c4b5fd",             link_text_color_dark="#c4b5fd",
    # chat bubbles (user bubble uses the accent colour)
    color_accent_soft="#2e1065",           color_accent_soft_dark="#2e1065",
    border_color_primary=PURPLE,           border_color_primary_dark=PURPLE,
    # buttons
    button_primary_background_fill=PURPLE, button_primary_background_fill_dark=PURPLE,
    button_primary_text_color="#ffffff",   button_primary_text_color_dark="#ffffff",
    button_secondary_background_fill=DARKER, button_secondary_background_fill_dark=DARKER,
    button_secondary_text_color="#ffffff", button_secondary_text_color_dark="#ffffff",
    button_secondary_border_color=PURPLE,  button_secondary_border_color_dark=PURPLE,
)

css = """
.gradio-container { background:#000 !important; }
.block { border-radius:10px !important; }
.block:focus-within { box-shadow:0 0 10px #8b5cf6; }

/* chat bubbles */
.message.user, [data-testid="user"]  { background:#2e1065 !important; color:#fff !important; border-color:#8b5cf6 !important; }
.message.bot,  [data-testid="bot"]   { background:#1a1a1a !important; color:#fff !important; border-color:#8b5cf6 !important; }
.message *, [data-testid="user"] *, [data-testid="bot"] * { color:#fff !important; }

/* file list rows and sizes */
.file-preview tr, .file-preview td { background:#0a0a0a !important; color:#fff !important; }
.file-preview tr:hover td { background:#1a1a1a !important; }
.file-preview a, .file-preview .download { color:#c4b5fd !important; }

/* gradient primary buttons (Send, Process Documents) */
button.primary {
    background: linear-gradient(90deg, #3b0764, #c4b5fd, #3b0764) !important;
    background-size: 100% 100% !important;
    border: none !important;
    color: #fff !important;
    font-weight: 700 !important;
    text-shadow: 0 1px 2px rgba(0,0,0,0.4);
    box-shadow: none !important;
    transition: transform 0.15s ease, box-shadow 0.2s ease !important;
    /* animation removed so the dark-light-dark gradient stays put */
}
button.primary:hover {
    transform: translateY(-2px) scale(1.02);
    box-shadow: none !important;
}
button.primary:active { transform: scale(0.98); }

@keyframes gradientShift {
    0%   { background-position: 0% 50%; }
    100% { background-position: 200% 50%; }
}
"""
# ---------- 2 + 3. Multiple file upload with a loading indicator ----------
def process_pdfs(files, progress=gr.Progress()):
    if not files:
        return "⚠️ Please upload at least one PDF."
    names = []
    for f in progress.tqdm(files, desc="Processing documents"):
        time.sleep(1)  # stand-in for real PDF processing
        names.append(f.name.split("/")[-1])
    return f"✅ Processed {len(names)} file(s):\n" + "\n".join(f"- {n}" for n in names)

def handle_chat(message, history):
    history = history or []
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": "This is a mock answer."})
    return history, ""

# ---------- 4. Chat history export ----------
def to_text(content):
    if isinstance(content, list):  # Gradio may send content as a list of parts
        return " ".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content)

def export_chat(history):
    if not history:
        return None
    path = os.path.join(tempfile.mkdtemp(), "chat_history.txt")
    with open(path, "w") as f:
        for m in history:
            f.write(f"{m['role'].upper()}: {to_text(m['content'])}\n\n")
    return path

with gr.Blocks(title="Step 5: Full Functioning UI") as demo:
    gr.Markdown("### Step 5: Connected UI")
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=300)
            user_input = gr.Textbox(placeholder="Ask a question about your document...",
                                    label="Your Question")
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat", variant="primary")
                export_btn = gr.DownloadButton("💾 Export Chat", value=None, variant="primary")
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDFs", file_types=[".pdf"], file_count="multiple")
            process_btn = gr.Button("🔄 Process Documents", variant="primary")
            status = gr.Textbox(label="Status", interactive=False, lines=4, max_lines=15)

    process_btn.click(process_pdfs, inputs=pdf_input, outputs=status)
    send_btn.click(handle_chat, [user_input, chatbot], [chatbot, user_input])
    user_input.submit(handle_chat, [user_input, chatbot], [chatbot, user_input])
    clear_btn.click(lambda: [], outputs=chatbot)
    chatbot.change(export_chat, inputs=chatbot, outputs=export_btn)

demo.launch(theme=theme, css=css)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6491333a913c9d6a46.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
